In [22]:
%load_ext autoreload
%autoreload 2

import csv, os, time
import numpy as np
import matplotlib.pyplot as plt
import librosa, librosa.display
from IPython.display import Audio

from shapesearch.audio import load_audio, time_grid
from shapesearch.features.loudness import loudness_curve, usable_range
from shapesearch.features.brightness import brightness_curve
from shapesearch.features.hits import detect_hits

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:

ROOT = ".."   # this notebook lives in notebooks/, the data in ../data


def read_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


manifest = {int(r["track_id"]): r for r in read_csv(f"{ROOT}/data/manifest.csv")}

ground_truth = {}   # optional: only used if you made ground_truth.csv
gt_path = f"{ROOT}/data/ground_truth.csv"
if os.path.exists(gt_path):
    for r in read_csv(gt_path):
        ground_truth.setdefault(int(r["track_id"]), []).append((float(r["time_s"]), r["label"]))
print(f"{len(manifest)} tracks in the manifest, {len(ground_truth)} with ground-truth labels")

25 tracks in the manifest, 0 with ground-truth labels


In [24]:
_cache = {}


def analyze(track_id):
    """Everything the extractor will compute for one track (cached)."""
    if track_id not in _cache:
        y, sr = load_audio(f"{ROOT}/{manifest[track_id]['file_path']}")
        L_fast = loudness_curve(y, sr, window_s=0.4)   # short window: hits and usable range
        _cache[track_id] = dict(
            sr=sr, duration=len(y) / sr, t=time_grid(len(y) / sr),
            L=loudness_curve(y, sr), B=brightness_curve(y, sr),
            hits=detect_hits(L_fast), usable=usable_range(L_fast),
        )
    return _cache[track_id]


LABEL_COLORS = {"build_start": "orange", "peak": "purple", "drop": "green"}


def review(track_id):
    """Spectrogram, loudness and brightness with hit points (red), usable range
    (gray = silent edges) and your hand labels (dashed, if you made any)."""
    r = analyze(track_id)
    t, start_s, end_s = r["t"], *r["usable"]
    y, sr = load_audio(f"{ROOT}/{manifest[track_id]['file_path']}")
    S = librosa.amplitude_to_db(np.abs(librosa.stft(y.mean(axis=1), n_fft=2048, hop_length=512)), ref=np.max)

    fig, axes = plt.subplots(3, 1, figsize=(13, 8))
    librosa.display.specshow(S, sr=sr, hop_length=512, x_axis="time", y_axis="log", ax=axes[0])
    axes[0].set_title(f"Track {track_id}: {manifest[track_id]['title']}  |  {r['duration']:.0f} s  |  {len(r['hits'])} hits")
    axes[1].plot(t, r["L"]); axes[1].set_ylabel("Loudness (LUFS)")
    axes[2].plot(t, r["B"]); axes[2].set_ylabel("Brightness (octaves)"); axes[2].set_xlabel("Time (s)")

    for ax in axes:
        ax.set_xlim(0, t[-1])
        for time_s, _ in r["hits"]:
            ax.axvline(time_s, color="red", lw=1.2)
    for ax in axes[1:]:
        ax.grid(alpha=0.3)
        ax.axvspan(0, start_s, color="gray", alpha=0.25)
        ax.axvspan(end_s, t[-1], color="gray", alpha=0.25)
    seen = set()
    for time_s, label in ground_truth.get(track_id, []):
        axes[1].axvline(time_s, color=LABEL_COLORS.get(label, "black"), ls="--",
                        label=None if label in seen else label)
        seen.add(label)
    if seen:
        axes[1].legend(loc="lower right")
    plt.tight_layout()
    plt.show()


def listen(track_id, start_s, end_s):
    """Play an excerpt (mono, 22 kHz so the notebook file stays small)."""
    y, sr = load_audio(f"{ROOT}/{manifest[track_id]['file_path']}")
    seg = y[int(start_s * sr): int(end_s * sr)].mean(axis=1)
    return Audio(librosa.resample(seg, orig_sr=sr, target_sr=22050), rate=22050)


def listen_to_hit(track_id, n=0, seconds_around=6):
    """Play the audio around the n-th detected hit (0 = first)."""
    time_s = analyze(track_id)["hits"][n][0]
    print(f"hit {n} at {time_s:.1f} s")
    return listen(track_id, max(0, time_s - seconds_around), time_s + seconds_around)

In [25]:
start = time.time()
print(f"{'id':>3}  {'title':<26}{'dur':>5}{'usable':>14}{'loudness (LUFS)':>20}{'brightness (oct)':>19}{'hits':>6}")
for tid in sorted(manifest):
    r = analyze(tid)
    s, e = r["usable"]
    usable = f"{s:.0f}-{e:.0f} s"
    loud = f"{r['L'].min():.0f} to {r['L'].max():.0f}"
    bright = f"{r['B'].min():.1f} to {r['B'].max():.1f}"
    title = manifest[tid]["title"][:25]
    print(f"{tid:>3}  {title:<26}{r['duration']:>5.0f}{usable:>14}{loud:>20}{bright:>19}{len(r['hits']):>6}")
print(f"sweep took {time.time() - start:.0f} s")

 id  title                       dur        usable     loudness (LUFS)   brightness (oct)  hits
  0  1st Contact - Duty Free     300       2-299 s           -59 to -8        9.6 to 12.6     1
  1  1st Contact - Uprooted      266       0-266 s          -26 to -10        9.4 to 12.8     1
  2  Adeline Yeo (HP) - Innova   218       0-213 s           -70 to -4        9.3 to 11.2     4
  3  BlackTrendMusic - Synth P   194       0-192 s           -69 to -6        8.7 to 13.1     3
  4  HoliznaCC0 - Mutant Club    131       0-125 s           -70 to -7        8.3 to 12.7     0
  5  I Could Use a Boost         187       0-179 s           -70 to -6        7.2 to 11.6     4
  6  JMHBM - 11 MPH              186       0-183 s          -70 to -10        9.0 to 12.3     5
  7  JMHBM - Flashy              144       0-143 s           -38 to -8        9.1 to 12.4     2
  8  JMHBM - Inhale              192       0-189 s          -70 to -10        8.6 to 12.5     6
  9  Jahzzar - Echoes            199    